# 🔬 Lab: LangGraph Multi-Agent Systems

Welcome to the interactive lab for the NCP-AAI Study Guide. This notebook covers key engineering concepts tested in the certification.

**Prerequisites:**
- NVIDIA API Key (`nvapi-...`)
- OpenAI API Key (optional, for RAGAS evaluation/tracing)


### Step 1: Define the Graph State
LangGraph uses explicit state objects passed between nodes.


In [ ]:
!pip install -qU langgraph langchain-core

from typing import TypedDict, Annotated, Sequence
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    current_agent: str
    research_data: str


### Step 2: Define Nodes and Edges


In [ ]:
from langgraph.graph import StateGraph, END

def researcher_node(state):
    print('--- Researcher Node Running ---')
    return {'research_data': 'Found 3 PDF documents on TensorRT limitats.', 'current_agent': 'summarizer'}

def summarizer_node(state):
    print('--- Summarizer Node Running ---')
    # In reality this calls an LLM
    return {'messages': [{'role': 'assistant', 'content': 'Summary complete: TensorRT requires Hopper or Ada for FP8.'}]}

workflow = StateGraph(AgentState)
workflow.add_node('researcher', researcher_node)
workflow.add_node('summarizer', summarizer_node)

# Define routing logic
def route_agent(state):
    if state.get('current_agent') == 'summarizer':
        return 'summarizer'
    return END

workflow.set_entry_point('researcher')
workflow.add_conditional_edges('researcher', route_agent)
workflow.add_edge('summarizer', END)

app = workflow.compile()
print('LangGraph compiled into DAG!')
